In [540]:
"""Fuente: https://www.desinventar.net/"""

'Fuente: https://www.desinventar.net/'

## Librerias necesarias

In [541]:
import pandas as pd

## Leer csv con datos de utilidad

In [542]:
df = pd.read_csv("Archivos creados/fichas.csv", low_memory=False) #Low_memory ignora la advertencia de de columnas mixtas
print(df.shape)

(71629, 64)


## Verificar sus columnas y extraer las necesarias

In [543]:
df.columns

Index(['serial', 'level0', 'level1', 'level2', 'name0', 'name1', 'name2',
       'evento', 'lugar', 'fechano', 'fechames', 'fechadia', 'muertos',
       'heridos', 'desaparece', 'afectados', 'vivdest', 'vivafec', 'otros',
       'fuentes', 'valorloc', 'valorus', 'fechapor', 'fechafec', 'hay_muertos',
       'hay_heridos', 'hay_deasparece', 'hay_afectados', 'hay_vivdest',
       'hay_vivafec', 'hay_otros', 'socorro', 'salud', 'educacion',
       'agropecuario', 'industrias', 'acueducto', 'alcantarillado', 'energia',
       'comunicaciones', 'causa', 'descausa', 'transporte', 'magnitud2',
       'nhospitales', 'nescuelas', 'nhectareas', 'cabezas', 'kmvias',
       'duracion', 'damnificados', 'evacuados', 'hay_damnificados',
       'hay_evacuados', 'hay_reubicados', 'reubicados', 'clave', 'glide',
       'defaultab', 'approved', 'latitude', 'longitude', 'uu_id',
       'di_comments'],
      dtype='object')

In [544]:
df = df[['name0', 'name1', 'name2', 'evento', "fechano", "fechames", "fechadia"]]
df["fecha"] = pd.to_datetime(
    {
        "year": df["fechano"],
        "month": df["fechames"],
        "day": df["fechadia"]
    }
)
df = df.drop(columns=[ "fechano", "fechames", "fechadia"])
df = df.rename(columns= {'name0': "Provincia",
                  'name1': "Cantón",
                  'name2': "Parroquia" })



df.head()

,Provincia,Cantón,Parroquia,evento,fecha
0,AZUAY,CUENCA,NaN,DESLIZAMIENTO,2000-04-29
1,SUCUMBIOS,SHUSHUFINDI,NaN,INCENDIO ESTRUCTURAL,2000-04-29
2,CARCHI,NaN,NaN,INUNDACIÓN,2000-04-26
3,CHIMBORAZO,COLTA,NaN,DESLIZAMIENTO,2000-04-25
4,PICHINCHA,QUITO,NaN,DESLIZAMIENTO,2000-04-21


## Filtrar unicamente por inundaciones

In [545]:
df_inundaciones = df[df["evento"] == "INUNDACIÓN"].reset_index(drop= True)
print(df_inundaciones.shape)
df_inundaciones.head()

(7684, 5)


,Provincia,Cantón,Parroquia,evento,fecha
0,CARCHI,NaN,NaN,INUNDACIÓN,2000-04-26
1,AZUAY,CUENCA,NaN,INUNDACIÓN,2000-04-13
2,PICHINCHA,RUMIÑAHUI,NaN,INUNDACIÓN,2000-03-23
3,PICHINCHA,RUMIÑAHUI,NaN,INUNDACIÓN,2000-03-13
4,GUAYAS,GUAYAQUIL,NaN,INUNDACIÓN,2000-03-08


## Ver valores nulos

In [546]:
df_inundaciones.isnull().sum()

Provincia       0
Cantón        107
Parroquia    1318
evento          0
fecha           0
dtype: int64

## Eliminamos Datos nulos en parroquias

In [547]:
#Eran demasiados datos por rellenar, asi que decidí eliminarlos
df_inundaciones = df_inundaciones.dropna(subset=["Parroquia"]).reset_index(drop=True)
df_inundaciones.isnull().sum()

Provincia    0
Cantón       0
Parroquia    0
evento       0
fecha        0
dtype: int64

In [548]:
#Eliminamos variables que ya no vamos a usar
print(df_inundaciones.shape)
df_inundaciones = df_inundaciones.drop(columns=["evento"])
df_inundaciones.head()

(6366, 5)


,Provincia,Cantón,Parroquia,fecha
0,NAPO,TENA,AHUANO,2008-09-22
1,ESMERALDAS,SAN LORENZO,CONCEPCION,2008-02-09
2,EL ORO,ARENILLAS,CARCABON,2008-03-18
3,EL ORO,ARENILLAS,CARCABON,2008-02-10
4,EL ORO,PASAJE,BUENAVISTA,2008-02-24


## Viendo fecha de incio y final de registros

In [549]:
df_fechas = df_inundaciones.copy()
df_fechas = df_fechas.sort_values(by="fecha").reset_index(drop=True)
print("Desde: ", df_fechas["fecha"].min().strftime("%Y-%m-%d"))
print("Hasta: ", df_fechas["fecha"].max().strftime("%Y-%m-%d"))

Desde:  2007-06-15
Hasta:  2023-12-30


## Normalizamos texto

In [550]:
import unicodedata
def normalizar(texto):

    if pd.isna(texto):
        return texto

    #Eliminar espacios al inicio y al final, y coloca en mayusculas
    texto = str(texto).strip().upper()

    # Eliminar tildes y diéresis
    texto = ''.join(
        c for c in unicodedata.normalize('NFD', texto)
        if unicodedata.category(c) != 'Mn'
    )

    # Eliminar espacios dobles
    texto = ' '.join(texto.split())

    return texto

df_inundaciones["Provincia"] = df_inundaciones["Provincia"].apply(normalizar)
df_inundaciones["Cantón"] = df_inundaciones["Cantón"].apply(normalizar)
df_inundaciones["Parroquia"] = df_inundaciones["Parroquia"].apply(normalizar)

## Contando Duplicados por Provincia, Cantón y Parroquia

In [551]:
#Esta linea solo sirve para visualizar la logica de lo que hara la siguiente celda
df_inundaciones.groupby(["Provincia","Cantón","Parroquia"]).size().reset_index()

,Provincia,Cantón,Parroquia,0
0,AZUAY,CAMILO PONCE ENRIQUEZ,CAMILO PONCE ENRIQUEZ,12
1,AZUAY,CAMILO PONCE ENRIQUEZ,EL CARMEN DE PUJILI,1
2,AZUAY,CHORDELEG,CHORDELEG,5
3,AZUAY,CHORDELEG,SAN MARTIN DE PUZHIO,1
4,AZUAY,CUENCA,BANOS,1
...,...,...,...,...
967,ZAMORA CHINCHIPE,ZAMORA,GUADALUPE,4
968,ZAMORA CHINCHIPE,ZAMORA,SABANILLA,3
969,ZAMORA CHINCHIPE,ZAMORA,SAN CARLOS DE LAS MINAS,7
970,ZAMORA CHINCHIPE,ZAMORA,TIMBARA,5


In [552]:
#Juntando duplicados, para crear columna de num_inundaciones, mediante provincia canton y parroquia
df_inundaciones_conteo_text = (
    df_inundaciones
    .groupby(["Provincia", "Cantón", "Parroquia"]) #Agrupamos por "Provincia", "Cantón" y "Parroquia" similares
    .size()#Contamos cuantas fila tiene cada grupo
    .reset_index(name="num_inundaciones")
)

print(df_inundaciones_conteo_text.shape)
df_inundaciones_conteo_text.head()

(972, 4)


,Provincia,Cantón,Parroquia,num_inundaciones
0,AZUAY,CAMILO PONCE ENRIQUEZ,CAMILO PONCE ENRIQUEZ,12
1,AZUAY,CAMILO PONCE ENRIQUEZ,EL CARMEN DE PUJILI,1
2,AZUAY,CHORDELEG,CHORDELEG,5
3,AZUAY,CHORDELEG,SAN MARTIN DE PUZHIO,1
4,AZUAY,CUENCA,BANOS,1


In [553]:
#Visualiando parroquias que hayan tenido mas de 10 inundaciones
df_inundaciones_conteo_text[df_inundaciones_conteo_text["num_inundaciones"] > 10]

,Provincia,Cantón,Parroquia,num_inundaciones
0,AZUAY,CAMILO PONCE ENRIQUEZ,CAMILO PONCE ENRIQUEZ,12
15,AZUAY,CUENCA,MOLLETURO,11
37,AZUAY,GUALACEO,GUALACEO,15
56,BOLIVAR,CALUMA,CALUMA,13
61,BOLIVAR,ECHEANDIA,ECHEANDIA,21
...,...,...,...,...
952,ZAMORA CHINCHIPE,EL PANGUI,EL PANGUI,17
962,ZAMORA CHINCHIPE,YANTZAZA,CHICANA,11
964,ZAMORA CHINCHIPE,YANTZAZA,YANTZAZA (YANZATZA),15
965,ZAMORA CHINCHIPE,ZAMORA,CUMBARATZA,18


## Clasificando riesgo de inundacion por cuartiles

In [554]:
#Clasificacion de riesgo por cuartiles

df_inundaciones_conteo_text_quartil = df_inundaciones_conteo_text.copy()

df_inundaciones_conteo_text_quartil["riesgo_inundacion"] = pd.qcut(
    df_inundaciones_conteo_text["num_inundaciones"],
    q= 3,
    labels=["Bajo", "Medio", "Alto"]#Convierte la columna a tipo de dato categorico
)
df_inundaciones_conteo_text_quartil.head(2)

,Provincia,Cantón,Parroquia,num_inundaciones,riesgo_inundacion
0,AZUAY,CAMILO PONCE ENRIQUEZ,CAMILO PONCE ENRIQUEZ,12,Alto
1,AZUAY,CAMILO PONCE ENRIQUEZ,EL CARMEN DE PUJILI,1,Bajo


In [555]:
#Distribucion de datos
df_inundaciones_conteo_text_quartil["riesgo_inundacion"].value_counts()

riesgo_inundacion
Bajo     389
Medio    298
Alto     285
Name: count, dtype: int64

In [556]:
#Rangos de clasificacion
df_inundaciones_conteo_text_quartil.groupby("riesgo_inundacion", observed=False)["num_inundaciones"].agg(
    ["min", "max"]
)

,min,max
riesgo_inundacion,,
Bajo,1,2
Medio,3,6
Alto,7,105


In [557]:
df_inundaciones_conteo_text_quartil[df_inundaciones_conteo_text_quartil["num_inundaciones"] > 100]

,Provincia,Cantón,Parroquia,num_inundaciones,riesgo_inundacion
357,GUAYAS,GUAYAQUIL,TARQUI,105,Alto


In [558]:
df_inundaciones_conteo_text_quartil[df_inundaciones_conteo_text_quartil["Parroquia"] == "TARQUI"]

,Provincia,Cantón,Parroquia,num_inundaciones,riesgo_inundacion
28,AZUAY,CUENCA,TARQUI,6,Medio
357,GUAYAS,GUAYAQUIL,TARQUI,105,Alto
597,MANABI,MANTA,TARQUI,15,Alto
762,PASTAZA,PASTAZA,TARQUI,4,Medio


In [559]:
df_inundaciones_conteo_text_quartil = df_inundaciones_conteo_text_quartil.rename(columns= {'Provincia': "DPA_DESPRO",
                                                                                              'Cantón': "DPA_DESCAN",
                                                                                              'Parroquia': "DPA_DESPAR",})

In [560]:
df_inundaciones_conteo_text_quartil.head(3)

,DPA_DESPRO,DPA_DESCAN,DPA_DESPAR,num_inundaciones,riesgo_inundacion
0,AZUAY,CAMILO PONCE ENRIQUEZ,CAMILO PONCE ENRIQUEZ,12,Alto
1,AZUAY,CAMILO PONCE ENRIQUEZ,EL CARMEN DE PUJILI,1,Bajo
2,AZUAY,CHORDELEG,CHORDELEG,5,Medio


In [561]:
df_inundaciones_conteo_text_quartil["riesgo_inundacion"].value_counts()

riesgo_inundacion
Bajo     389
Medio    298
Alto     285
Name: count, dtype: int64

## Dataset Con variables independientes

In [562]:
df_independientes= pd.read_csv("data.csv")
df_independientes= df_independientes.drop(columns= ['Unnamed: 0'])
print(df_independientes.shape)
df_independientes.head(2)

(1050, 10)


,DPA_PARROQ,DPA_DESPAR,altura_mean,altura_min,altura_max,pendiente_mean,distancia_rios,precipitaciones_mean,DPA_DESPRO,DPA_DESCAN
0,010150,CUENCA,2548.790629,2331,2863,89.428921,1110.917269,73.119566,AZUAY,CUENCA
1,010151,BAÑOS,3492.725017,2569,4136,89.902887,819.428420,78.623622,AZUAY,CUENCA


## Normalizando variables del dataset de independientes

In [563]:
df_independientes["DPA_DESPAR"] = df_independientes["DPA_DESPAR"].apply(normalizar)
df_independientes["DPA_DESCAN"] = df_independientes["DPA_DESCAN"].apply(normalizar)
df_independientes["DPA_DESPRO"] = df_independientes["DPA_DESPRO"].apply(normalizar)

## Uniendo datasets

In [564]:
df_final_text =  df_independientes.merge(
    df_inundaciones_conteo_text_quartil[
        ["DPA_DESPRO", "DPA_DESCAN","DPA_DESPAR", "num_inundaciones", "riesgo_inundacion"]
    ],
    on=["DPA_DESPRO", "DPA_DESCAN","DPA_DESPAR"],
    how="left" #Conservar todas las filas de el dataset de la izquierda (df_independientes)
)
print(df_final_text.shape)
df_final_text.head(3)

(1050, 12)


,DPA_PARROQ,DPA_DESPAR,altura_mean,altura_min,altura_max,pendiente_mean,distancia_rios,precipitaciones_mean,DPA_DESPRO,DPA_DESCAN,num_inundaciones,riesgo_inundacion
0,010150,CUENCA,2548.790629,2331,2863,89.428921,1110.917269,73.119566,AZUAY,CUENCA,NaN,NaN
1,010151,BANOS,3492.725017,2569,4136,89.902887,819.428420,78.623622,AZUAY,CUENCA,1.0,Bajo
2,010152,CUMBE,3006.112686,2620,3477,89.977973,1149.323612,65.115401,AZUAY,CUENCA,1.0,Bajo


In [565]:
df_final_text.isnull().sum()

DPA_PARROQ                0
DPA_DESPAR                0
altura_mean               0
altura_min                0
altura_max                0
pendiente_mean            0
distancia_rios            0
precipitaciones_mean      8
DPA_DESPRO                0
DPA_DESCAN                0
num_inundaciones        451
riesgo_inundacion       451
dtype: int64

In [566]:
#Se rellenan los valores nulos
df_final_text["riesgo_inundacion"]= df_final_text["riesgo_inundacion"].cat.add_categories("Nulo")
df_final_text["riesgo_inundacion"]= df_final_text["riesgo_inundacion"].fillna("Nulo")

df_final_text["num_inundaciones"]= df_final_text["num_inundaciones"].fillna(0)

In [567]:
df_final_text.isnull().sum()

DPA_PARROQ              0
DPA_DESPAR              0
altura_mean             0
altura_min              0
altura_max              0
pendiente_mean          0
distancia_rios          0
precipitaciones_mean    8
DPA_DESPRO              0
DPA_DESCAN              0
num_inundaciones        0
riesgo_inundacion       0
dtype: int64

In [568]:
df_final_text.shape

(1050, 12)

In [569]:
df_final_text.to_csv("dataset.csv")